##Jai Ganesha

# 00 — Project Setup, Environment, and Checkpoint Loading

Goal: Mount Google Drive, define all project paths, clone YOLO-World and SAM-2 repos,
install dependencies, verify the GPU, and confirm both pretrained checkpoints can load.

This notebook runs FIRST before every session. It should be short, stable, and never contain
dataset, training, or evaluation logic.

Expected output by end of this notebook:
- Google Drive mounted
- All project paths defined and directory tree created
- YOLO-World and SAM-2 repos cloned
- All packages installed (pinned versions)
- GPU verified
- Both model checkpoints load without errors
- config.json saved to Drive for reuse

In [1]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Mount Google Drive and define all project paths
# ─────────────────────────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

import os

# ── Project root on your Drive ────────────────────────────────
PROJECT_ROOT = "/content/drive/MyDrive/DLCV_OV_Analytics"

# ── Dataset paths (raw = shortcuts to shared Drive folders) ───
# These are NOT created by makedirs — they already exist as shortcuts
RAW_DATA       = os.path.join(PROJECT_ROOT, "datasets/raw")
RAW_VISDRONE   = os.path.join(RAW_DATA, "VisDrone")       # match your exact shortcut name
RAW_COCO       = os.path.join(RAW_DATA, "COCO")           # match your exact shortcut name
RAW_YTVIS      = os.path.join(RAW_DATA, "Youtube VIS")    # match your exact shortcut name

# ── Processed data paths ──────────────────────────────────────
PROC_DATA      = os.path.join(PROJECT_ROOT, "datasets/processed")
PROC_VISDRONE  = os.path.join(PROC_DATA, "visdrone")
PROC_COCO      = os.path.join(PROC_DATA, "coco")
PROC_YTVIS     = os.path.join(PROC_DATA, "youtube_vis")
UNIFIED_DIR    = os.path.join(PROC_DATA, "unified")

# ── Checkpoints ───────────────────────────────────────────────
CKPT_DIR       = os.path.join(PROJECT_ROOT, "checkpoints")
CKPT_YOLOWORLD = os.path.join(CKPT_DIR, "yoloworld")
CKPT_SAM2      = os.path.join(CKPT_DIR, "sam2")

# ── Outputs ───────────────────────────────────────────────────
OUTPUT_DIR     = os.path.join(PROJECT_ROOT, "outputs")
VIZ_DIR        = os.path.join(OUTPUT_DIR, "visualizations")
VIDEO_DIR      = os.path.join(OUTPUT_DIR, "videos")
METRICS_DIR    = os.path.join(OUTPUT_DIR, "metrics")
TABLES_DIR     = os.path.join(OUTPUT_DIR, "tables")

# ── Project structure ─────────────────────────────────────────
NOTEBOOKS_DIR  = os.path.join(PROJECT_ROOT, "notebooks")
CONFIGS_DIR    = os.path.join(PROJECT_ROOT, "configs")
LOGS_DIR       = os.path.join(PROJECT_ROOT, "logs")

# ── src/ module structure ─────────────────────────────────────
SRC_DIR        = os.path.join(PROJECT_ROOT, "src")
SRC_DATA       = os.path.join(SRC_DIR, "data")
SRC_MODELS     = os.path.join(SRC_DIR, "models")
SRC_VIDEO      = os.path.join(SRC_DIR, "video")
SRC_EVAL       = os.path.join(SRC_DIR, "eval")
SRC_UTILS      = os.path.join(SRC_DIR, "utils")

# ── Processed VisDrone subfolders (pre-created for notebook 02) ──
PROC_VD_IMG_TRAIN = os.path.join(PROC_VISDRONE, "images/train")
PROC_VD_IMG_VAL   = os.path.join(PROC_VISDRONE, "images/val")
PROC_VD_IMG_TEST  = os.path.join(PROC_VISDRONE, "images/test")
PROC_VD_LBL_TRAIN = os.path.join(PROC_VISDRONE, "labels/train")
PROC_VD_LBL_VAL   = os.path.join(PROC_VISDRONE, "labels/val")
PROC_VD_LBL_TEST  = os.path.join(PROC_VISDRONE, "labels/test")

# ── Repo paths (local Colab only, NOT on Drive) ───────────────
YOLOWORLD_REPO = "/content/YOLO-World"
SAM2_REPO      = "/content/sam2"

# ── Create all directories that need to be made ───────────────
# RAW_* paths are intentionally excluded — they are shortcuts that already exist
all_dirs = [
    # datasets/raw parent (but NOT the shortcut subfolders inside it)
    RAW_DATA,

    # processed data
    PROC_DATA,
    PROC_VISDRONE, PROC_COCO, PROC_YTVIS, UNIFIED_DIR,

    # processed VisDrone image/label subfolders
    PROC_VD_IMG_TRAIN, PROC_VD_IMG_VAL, PROC_VD_IMG_TEST,
    PROC_VD_LBL_TRAIN, PROC_VD_LBL_VAL, PROC_VD_LBL_TEST,

    # checkpoints
    CKPT_DIR, CKPT_YOLOWORLD, CKPT_SAM2,

    # outputs
    OUTPUT_DIR, VIZ_DIR, VIDEO_DIR, METRICS_DIR, TABLES_DIR,

    # project structure
    NOTEBOOKS_DIR, CONFIGS_DIR, LOGS_DIR,

    # src modules
    SRC_DIR, SRC_DATA, SRC_MODELS, SRC_VIDEO, SRC_EVAL, SRC_UTILS,
]

for d in all_dirs:
    os.makedirs(d, exist_ok=True)

# ── Verify raw dataset shortcuts are reachable ────────────────
print("Verifying raw dataset shortcuts...")
for name, path in [("VisDrone", RAW_VISDRONE), ("COCO", RAW_COCO), ("Youtube VIS", RAW_YTVIS)]:
    status = "✅" if os.path.exists(path) else "❌ NOT FOUND — check shortcut name"
    print(f"   {status}  {name}: {path}")

print("\n✅ All directories created.")
print(f"   Project root : {PROJECT_ROOT}")
print(f"   Raw data     : {RAW_DATA}")
print(f"   Checkpoints  : {CKPT_DIR}")
print(f"   Outputs      : {OUTPUT_DIR}")
print(f"   src/         : {SRC_DIR}")

Mounted at /content/drive
Verifying raw dataset shortcuts...
   ✅  VisDrone: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/VisDrone
   ✅  COCO: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO
   ✅  Youtube VIS: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/Youtube VIS

✅ All directories created.
   Project root : /content/drive/MyDrive/DLCV_OV_Analytics
   Raw data     : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw
   Checkpoints  : /content/drive/MyDrive/DLCV_OV_Analytics/checkpoints
   Outputs      : /content/drive/MyDrive/DLCV_OV_Analytics/outputs
   src/         : /content/drive/MyDrive/DLCV_OV_Analytics/src


In [2]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Clone YOLO-World and SAM-2 repos (skip if already done)
# ─────────────────────────────────────────────────────────────

import subprocess

def run(cmd):
    """Helper to run shell commands and print output cleanly."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"❌ ERROR:\n{result.stderr}")
    else:
        print(result.stdout if result.stdout.strip() else "✅ Done")
    return result.returncode

# ── Clone YOLO-World ─────────────────────────────────────────
if not os.path.exists(YOLOWORLD_REPO):
    print("Cloning YOLO-World...")
    run(f"git clone https://github.com/AILab-CVC/YOLO-World.git {YOLOWORLD_REPO}")
else:
    print("✅ YOLO-World already cloned.")

# ── Clone SAM-2 ──────────────────────────────────────────────
if not os.path.exists(SAM2_REPO):
    print("Cloning SAM-2...")
    run(f"git clone https://github.com/facebookresearch/sam2.git {SAM2_REPO}")
else:
    print("✅ SAM-2 already cloned.")

print(f"\nYOLO-World repo : {YOLOWORLD_REPO}")
print(f"SAM-2 repo      : {SAM2_REPO}")

Cloning YOLO-World...
✅ Done
Cloning SAM-2...
✅ Done

YOLO-World repo : /content/YOLO-World
SAM-2 repo      : /content/sam2


In [3]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Install dependencies
# ultralytics is pinned to 8.3.x — versions 8.4.11+ have a known
# bug where `from ultralytics import YOLO` fails on Python 3.12.
# See: https://github.com/ultralytics/ultralytics/issues/23574
# ─────────────────────────────────────────────────────────────

import subprocess, sys

def install(pkg, label=None):
    label = label or pkg
    print(f"  Installing {label}...", end=" ", flush=True)
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    print("✅" if r.returncode == 0 else f"❌\n{r.stderr[-200:]}")

print("Installing packages...")
install("ultralytics==8.3.145",   "ultralytics (pinned 8.3.145)")  # 8.4.11+ breaks YOLO import
install("supervision",            "supervision")
install("albumentations",         "albumentations")
install("pycocotools",            "pycocotools")
install("einops",                 "einops")
install("opencv-python-headless", "opencv-headless")
install("hydra-core",             "hydra-core")
install("iopath",                 "iopath")
install("submitit",               "submitit")

install("git+https://github.com/ultralytics/CLIP.git", "CLIP (pre-install)")

# ── Add SAM-2 to sys.path ONLY — NOT YOLO-World ──────────────
# Adding YOLO-World to sys.path causes its internal ultralytics/
# subfolder to shadow the pip-installed ultralytics package.
if SAM2_REPO not in sys.path:
    sys.path.insert(0, SAM2_REPO)
    print(f"  Added to sys.path: {SAM2_REPO} ✅")

# ── Force Python to reload ultralytics from the newly installed version ──
import importlib
import pkgutil

# Remove any stale ultralytics modules from the cache
stale = [m for m in sys.modules if m == "ultralytics" or m.startswith("ultralytics.")]
for m in stale:
    del sys.modules[m]

# ── Verify all imports ────────────────────────────────────────
print("\nVerifying imports...")
import torch
print(f"  torch        : {torch.__version__} | CUDA: {torch.cuda.is_available()}")

try:
    import ultralytics
    from ultralytics import YOLO
    print(f"  ultralytics  : {ultralytics.__version__} ✅  (YOLO importable)")
except Exception as e:
    print(f"  ultralytics  : ❌ {e}")

try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    from sam2.sam2_video_predictor import SAM2VideoPredictor
    print("  SAM-2        : ✅")
except Exception as e:
    print(f"  SAM-2        : ❌ {e}")

try:
    import cv2
    print(f"  opencv       : {cv2.__version__} ✅")
except Exception as e:
    print(f"  opencv       : ❌ {e}")

print("\n✅ Cell 3 complete.")

Installing packages...
  Installing ultralytics (pinned 8.3.145)... ✅
  Installing supervision... ✅
  Installing albumentations... ✅
  Installing pycocotools... ✅
  Installing einops... ✅
  Installing opencv-headless... ✅
  Installing hydra-core... ✅
  Installing iopath... ✅
  Installing submitit... ✅
  Installing CLIP (pre-install)... ✅
  Added to sys.path: /content/sam2 ✅

Verifying imports...
  torch        : 2.10.0+cu128 | CUDA: True
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  ultralytics  : 8.3.145 ✅  (YOLO importable)
  SAM-2        : ✅
  opencv       : 4.13.0 ✅

✅ Cell 3 complete.


In [4]:
# ── Post-install import check ─────────────────────────────────
import torch, cv2, ultralytics, PIL, yaml, pandas, albumentations

print("Import check:")
print(f"  torch        : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
print(f"  ultralytics  : {ultralytics.__version__}")
print(f"  opencv       : {cv2.__version__}")
print(f"  pillow       : {PIL.__version__}")
print(f"  albumentations: {albumentations.__version__}")
print(f"  pandas       : {pandas.__version__}")
print(f"  yaml         : {yaml.__version__}")

# Confirm YOLO-World and SAM-2 are importable
try:
    from ultralytics import YOLO
    print("  YOLO-World   : ✅ importable")
except Exception as e:
    print(f"  YOLO-World   : ❌ {e}")

try:
    import sys
    sys.path.insert(0, "/content/sam2")
    from sam2.build_sam import build_sam2
    print("  SAM-2        : ✅ importable")
except Exception as e:
    print(f"  SAM-2        : ❌ {e}")

Import check:
  torch        : 2.10.0+cu128  |  CUDA: True
  ultralytics  : 8.3.145
  opencv       : 4.13.0
  pillow       : 11.3.0
  albumentations: 2.0.8
  pandas       : 2.2.2
  yaml         : 6.0.3
  YOLO-World   : ✅ importable
  SAM-2        : ✅ importable


In [5]:
# ─────────────────────────────────────────────────────────────
# CELL 4: Verify GPU availability and CUDA version
# ─────────────────────────────────────────────────────────────

import torch

print("=" * 50)
print(f"PyTorch version      : {torch.__version__}")
print(f"CUDA available       : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version         : {torch.version.cuda}")
    print(f"GPU name             : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory (total)   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU memory (reserved): {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    DEVICE = "cuda"
else:
    print("⚠️  No GPU found! Go to Runtime > Change runtime type > T4 GPU")
    DEVICE = "cpu"

print(f"\nActive device        : {DEVICE}")
print("=" * 50)

PyTorch version      : 2.10.0+cu128
CUDA available       : True
CUDA version         : 12.8
GPU name             : NVIDIA A100-SXM4-40GB
GPU memory (total)   : 42.41 GB
GPU memory (reserved): 0.00 GB

Active device        : cuda


In [6]:
# ─────────────────────────────────────────────────────────────
# CELL 5: Download pretrained checkpoints to Drive
# Saved to Drive so they survive Colab session resets.
# ─────────────────────────────────────────────────────────────

import subprocess
from pathlib import Path

# ── YOLO-World: download via Ultralytics (most reliable method) ──
print("Downloading YOLO-World L pretrained weights via Ultralytics...")
YW_CKPT_PATH = os.path.join(CKPT_YOLOWORLD, "yolov8l-world.pt")

if Path(YW_CKPT_PATH).exists():
    size_mb = Path(YW_CKPT_PATH).stat().st_size / 1e6
    print(f"✅ YOLO-World already exists ({size_mb:.1f} MB): {YW_CKPT_PATH}")
else:
    r = subprocess.run(
        [sys.executable, "-c",
         f"from ultralytics import YOLO; import shutil; "
         f"m=YOLO('yolov8l-world.pt'); "
         f"shutil.copy('yolov8l-world.pt', '{YW_CKPT_PATH}')"],
        capture_output=True, text=True
    )
    if Path(YW_CKPT_PATH).exists():
        size_mb = Path(YW_CKPT_PATH).stat().st_size / 1e6
        print(f"✅ YOLO-World saved ({size_mb:.1f} MB): {YW_CKPT_PATH}")
    else:
        print(f"❌ YOLO-World download failed.\n{r.stderr[-300:]}")

# ── SAM-2: download directly from Meta's CDN ─────────────────
import urllib.request

SAM2_CKPT_URL  = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
SAM2_CKPT_PATH = os.path.join(CKPT_SAM2, "sam2.1_hiera_large.pt")

if Path(SAM2_CKPT_PATH).exists():
    size_mb = Path(SAM2_CKPT_PATH).stat().st_size / 1e6
    print(f"✅ SAM-2 already exists ({size_mb:.1f} MB): {SAM2_CKPT_PATH}")
else:
    print("⬇️  Downloading SAM-2.1 Large (~900 MB)...")
    try:
        urllib.request.urlretrieve(SAM2_CKPT_URL, SAM2_CKPT_PATH)
        size_mb = Path(SAM2_CKPT_PATH).stat().st_size / 1e6
        print(f"✅ SAM-2 saved ({size_mb:.1f} MB): {SAM2_CKPT_PATH}")
    except Exception as e:
        print(f"❌ SAM-2 download failed: {e}")

# ── Verify SAM-2 config file is accessible ───────────────────
sam2_cfg = os.path.join(SAM2_REPO, "sam2/configs/sam2.1/sam2.1_hiera_l.yaml")
if os.path.exists(sam2_cfg):
    print(f"✅ SAM-2 config found: {sam2_cfg}")
else:
    print(f"⚠️  SAM-2 config not found at: {sam2_cfg}")
    print("   Try: sam2/configs/sam2.1/sam2.1_hiera_l.yaml inside the cloned repo")

print(f"\nYOLO-World ckpt : {YW_CKPT_PATH}")
print(f"SAM-2 ckpt      : {SAM2_CKPT_PATH}")

✅ YOLO-World already exists (95.7 MB): /content/drive/MyDrive/DLCV_OV_Analytics/checkpoints/yoloworld/yolov8l-world.pt
✅ SAM-2 already exists (898.1 MB): /content/drive/MyDrive/DLCV_OV_Analytics/checkpoints/sam2/sam2.1_hiera_large.pt
✅ SAM-2 config found: /content/sam2/sam2/configs/sam2.1/sam2.1_hiera_l.yaml

YOLO-World ckpt : /content/drive/MyDrive/DLCV_OV_Analytics/checkpoints/yoloworld/yolov8l-world.pt
SAM-2 ckpt      : /content/drive/MyDrive/DLCV_OV_Analytics/checkpoints/sam2/sam2.1_hiera_large.pt


In [7]:
# ─────────────────────────────────────────────────────────────
# CELL 6: Verify both checkpoints load and run a forward pass
# ─────────────────────────────────────────────────────────────

import torch, sys, numpy as np

# ── Test YOLO-World ───────────────────────────────────────────
print("Testing YOLO-World checkpoint...")
try:
    from ultralytics import YOLO
    yw_model = YOLO(YW_CKPT_PATH)
    dummy = np.zeros((64, 64, 3), dtype=np.uint8)
    yw_model.set_classes(["person", "car"])
    results = yw_model.predict(dummy, verbose=False)
    print(f"✅ YOLO-World loaded and forward pass OK")
    del yw_model, results
    torch.cuda.empty_cache()
except Exception as e:
    print(f"❌ YOLO-World verification failed: {e}")

# ── Test SAM-2 ────────────────────────────────────────────────
print("\nTesting SAM-2 checkpoint...")
try:
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor

    sam2_cfg_relative = "configs/sam2.1/sam2.1_hiera_l.yaml"
    sam2_model = build_sam2(sam2_cfg_relative, SAM2_CKPT_PATH, device=DEVICE)
    predictor  = SAM2ImagePredictor(sam2_model)
    print(f"✅ SAM-2 loaded OK on {DEVICE}")
    del sam2_model, predictor
    torch.cuda.empty_cache()
except Exception as e:
    print(f"❌ SAM-2 verification failed: {e}")

print("\n✅ Checkpoint verification complete.")

Testing YOLO-World checkpoint...


100%|████████████████████████████████████████| 338M/338M [00:00<00:00, 499MiB/s]


✅ YOLO-World loaded and forward pass OK

Testing SAM-2 checkpoint...
✅ SAM-2 loaded OK on cuda

✅ Checkpoint verification complete.


In [8]:
# ─────────────────────────────────────────────────────────────
# CELL 7: Save global config.json to Drive
# Every subsequent notebook loads this instead of redefining paths.
# ─────────────────────────────────────────────────────────────

import json

config = {
    "project_root"      : PROJECT_ROOT,
    "raw_data"          : RAW_DATA,
    "raw_visdrone"      : RAW_VISDRONE,
    "raw_coco"          : RAW_COCO,
    "raw_ytvis"         : RAW_YTVIS,
    "proc_data"         : PROC_DATA,
    "proc_visdrone"     : PROC_VISDRONE,
    "proc_coco"         : PROC_COCO,
    "proc_ytvis"        : PROC_YTVIS,
    "unified_dir"       : UNIFIED_DIR,
    "proc_vd_img_train" : PROC_VD_IMG_TRAIN,
    "proc_vd_img_val"   : PROC_VD_IMG_VAL,
    "proc_vd_img_test"  : PROC_VD_IMG_TEST,
    "proc_vd_lbl_train" : PROC_VD_LBL_TRAIN,
    "proc_vd_lbl_val"   : PROC_VD_LBL_VAL,
    "proc_vd_lbl_test"  : PROC_VD_LBL_TEST,
    "ckpt_dir"          : CKPT_DIR,
    "ckpt_yoloworld"    : CKPT_YOLOWORLD,
    "ckpt_sam2"         : CKPT_SAM2,
    "yw_ckpt_path"      : YW_CKPT_PATH,
    "sam2_ckpt_path"    : SAM2_CKPT_PATH,
    "output_dir"        : OUTPUT_DIR,
    "viz_dir"           : VIZ_DIR,
    "video_dir"         : VIDEO_DIR,
    "metrics_dir"       : METRICS_DIR,
    "tables_dir"        : TABLES_DIR,
    "notebooks_dir"     : NOTEBOOKS_DIR,
    "configs_dir"       : CONFIGS_DIR,
    "logs_dir"          : LOGS_DIR,
    "src_dir"           : SRC_DIR,
    "src_data"          : SRC_DATA,
    "src_models"        : SRC_MODELS,
    "src_video"         : SRC_VIDEO,
    "src_eval"          : SRC_EVAL,
    "src_utils"         : SRC_UTILS,
    "yoloworld_repo"    : YOLOWORLD_REPO,
    "sam2_repo"         : SAM2_REPO,
    "device"            : DEVICE,
}

config_path = os.path.join(CONFIGS_DIR, "config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ config.json saved: {config_path}")
print(f"   Total keys: {len(config)}")
print(json.dumps(config, indent=2))

✅ config.json saved: /content/drive/MyDrive/DLCV_OV_Analytics/configs/config.json
   Total keys: 38
{
  "project_root": "/content/drive/MyDrive/DLCV_OV_Analytics",
  "raw_data": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw",
  "raw_visdrone": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/VisDrone",
  "raw_coco": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/COCO",
  "raw_ytvis": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/Youtube VIS",
  "proc_data": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed",
  "proc_visdrone": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdrone",
  "proc_coco": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/coco",
  "proc_ytvis": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/youtube_vis",
  "unified_dir": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/unified",
  "proc_vd_img_train": "/content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/visdro